In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Integer Square Root

In this notebook we work with very big numbers.  Therefore, we use the module `Z` of the library Zarith, which implements integers of arbitrary size.  Their type is `Z.t`.

In [ ]:
#require "zarith.top";;

The function `isqrt` takes one argument:
- `n` is a natural number.

It returns the largest natural number $r$ such that
$r^2 \leq n$, i.e. we have
$$ \texttt{isqrt}(n) := \max\bigl(\{ r \in \mathbb{N} \mid r^2 \leq n \}\bigr). $$

**Implementation:** The result is computed recursively via a *divide-and-conquer* algorithm as follows:
1. $\texttt{isqrt}(0) = 0$.
2. $\bigl(2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4) + 1\bigr)^2 \leq n \rightarrow \texttt{isqrt}(n) = 2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4) + 1$.
3. $\bigl(2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4) + 1\bigr)^2 > n \rightarrow \texttt{isqrt}(n) = 2 \cdot \texttt{isqrt}(n \,\texttt{/}\, 4)$.

Here, $n \,\texttt{/}\, 4$ denotes the integer division of $n$ by $4$.

Inside the *local open* `Z.(...)`, the arithmetic operators `*`, `+`, `/`, and `**` refer to the integers of type `Z.t`, `~$` converts an `int` into a `Z.t`, and `one` is the number $1$.  The comparison operators are not redefined there.  Therefore, we compare numbers of type `Z.t` with the functions `Z.equal` ($=$) and `Z.leq` ($\leq$).

In [ ]:
let rec isqrt (n : Z.t) : Z.t =
  if Z.equal n Z.zero then
    Z.zero
  else
    let r = isqrt Z.(n / ~$4) in
    if Z.leq Z.((~$2 * r + one) ** 2) n then
      Z.(~$2 * r + one)
    else
      Z.(~$2 * r)

The function `Z.to_string` converts a number of type `Z.t` into a string.

In [ ]:
for n = 0 to 17 do
  Printf.printf "isqrt(%d) = %s\n" n (Z.to_string (isqrt (Z.of_int n)))
done

In order to test our implementation more thoroughly we use random numbers.  The function call `Random.init 0` initializes the random number generator with the *seed* `0`, so that the tests are reproducible.  The function call `Z.random_bits k` returns a random number $x$ such that $0 \leq x < 2^k$.

The function `run_tests` takes one argument:
- `no_tests` is the number of tests that are to be performed.

The function generates `no_tests` random integers `n` such that $0 \leq n < 2^{64}$ and tests whether
`isqrt n` is the *integer square root* of `n` in each case.  If a test fails, it raises an exception: The function `failwith` raises the exception `Failure` with the given message.  The comparison $(r + 1)^2 > n$ is written as `Z.lt n Z.((r + one) ** 2)`, since `Z.lt` is the comparison $<$.

In [ ]:
let run_tests (no_tests : int) : unit =
  Random.init 0;
  for _ = 1 to no_tests do
    let n = Z.random_bits 64 in
    let r = isqrt n in
    if not (Z.leq Z.(r * r) n && Z.lt n Z.((r + one) ** 2)) then
      failwith (Printf.sprintf "Error: %s != isqrt(%s)" (Z.to_string r) (Z.to_string n))
  done

The function `time`, which is defined in the file `setup.ml`, takes two arguments:
- `label` is a string,
- `f` is a function whose only argument is `()`.

It computes `f ()`, prints `label` together with the time this computation took, and returns the result of `f ()`.  The computation to be measured is wrapped into an anonymous function `fun () -> ...`, so that it is only executed when `time` calls `f`.

In [ ]:
time "run_tests" (fun () -> run_tests 1_000_000)

The function `sqrt2` takes one argument:
- `k` is a natural number.

It returns a string containing $\sqrt{2}$ up to `k` decimal places.

**Implementation:** We have
$$\sqrt{2 \cdot 10^{2 \cdot k}} = \sqrt{2} \cdot 10^k.$$
Hence the digits of $\texttt{isqrt}(2 \cdot 10^{2 \cdot k})$ are the first $k+1$ digits of $\sqrt{2}$.  We only have to insert a decimal point after the first digit.  The exponent `e` is computed outside of `Z.(...)`, since inside, the operator `*` multiplies numbers of type `Z.t`, not of type `int`.  The function `Z.to_string` converts a number into the string of its decimal digits and `String.sub s i l` returns the substring of `s` that starts at index `i` and has the length `l`.

In [ ]:
let sqrt2 (k : int) : string =
  let e = 2 * k in
  let s = Z.to_string (isqrt Z.(~$2 * ~$10 ** e)) in
  String.sub s 0 1 ^ "." ^ String.sub s 1 (String.length s - 1)

Let us compute the first $10\,000$ digits of $\sqrt{2\,}$.  The depth of the recursion is about $\log_4\bigl(2 \cdot 10^{20\,000}\bigr) \approx 33\,000$.  Every recursive call needs some memory on the *stack*.  In <samp>OCaml</samp> 5, the stack grows automatically up to a limit of one gigabyte.  Hence, unlike in Python, we do not have to increase a recursion limit.

In [ ]:
print_endline (time "sqrt2" (fun () -> sqrt2 10_000))

You can compare this with the results shown on https://catonmat.net/tools/generate-sqrt2-digits.